In [ ]:
function range(n: number): number[] {
    return Array.from({ length: n }, (_, i) => i);
}

In [ ]:
range(5)

# Sudoku

The sudoku we want to solve is shown here:
    <img src="sudoku.png" width="40%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The function `create_puzzle` returns a representation of this puzzle as an array of arrays.

In [ ]:
type CellValue = number | "*";

function create_puzzle(): CellValue[][] {
    return [ ["*",  3 ,  9 , "*", "*", "*", "*", "*",  7 ], 
             ["*", "*", "*",  7 , "*", "*",  4 ,  9 ,  2 ],
             ["*", "*", "*", "*",  6 ,  5 , "*",  8 ,  3 ],
             ["*", "*", "*",  6 , "*",  3 ,  2 ,  7 , "*"],
             ["*", "*", "*", "*",  4 , "*",  8 , "*", "*"],
             [ 5 ,  6 , "*", "*", "*", "*", "*", "*", "*"],
             ["*", "*",  5 ,  2 , "*",  9 , "*", "*",  1 ],
             ["*",  2 ,  1 , "*", "*", "*", "*",  4 , "*"],
             [ 7 , "*", "*", "*", "*", "*",  5 , "*", "*"]
           ];
}

We are going to solve this puzzle with the help of the constraint solver `Z3`.

In [ ]:
import { init, Arith, Bool } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

The function `constraints_from_puzzle` takes one argument:
* `Variables` is a matrix of `Z3` variables.  

   For `row`$\in \{0,8\}$ and `col`$\in \{0,8\}$ the variable `Variables[row][col]` specifies the number that is placed in the specified row and column.

It returns a set of constraints specifying that the variables corresponding to numbers that are already set in the given Sudoku take the specified values.

In [ ]:
function constraints_from_puzzle(Variables: Arith[][]): Bool[] {
    const Puzzle = create_puzzle();
    // your code here
}

In [ ]:
const Variables: Arith[][] = 
    range(9).map(row => range(9).map(col => Z3.Int.const(`V${row + 1}${col + 1}`)));

The function `all_constraints` returns a CSP that encodes the given sudoku as a CSP.

In [ ]:
function all_constraints(Variables: Arith[][]): Bool[] {
    const constraints    = constraints_from_puzzle(Variables);
    const rowConstraints = range(9).map(row => Z3.Distinct(...Variables[row]));
    const colConstraints = "your code here"
    const squareConstraints = "your code here"

    const boundConstraints = "your code here"
    
    return [
        ...constraints,
        ...rowConstraints,
        ...colConstraints,
        ...squareConstraints,
        ...boundConstraints
    ];
}

I have got 217 constraints.

In [ ]:
all_constraints(Variables).length

The function `solve()` computes a solution to the given problem and returns this solution.

In [ ]:
import { RecursiveMap as Map } from 'recursive-set';

async function solve(): Promise<Map<string, number> | undefined> {
    const Variables: Arith[][] = range(9).map(row => 
        range(9).map(col => Z3.Int.const(`V${row + 1}${col + 1}`))
    );
    
    const S = new Z3.Solver();
    S.add(...all_constraints(Variables));
    
    const result = await S.check();
    if (result == 'sat') {
        const M = S.model();
        const Solution = new Map<string, number>();
        range(9).forEach(row => 
            range(9).forEach(col => {
                const varName = `V${row + 1}${col + 1}`;
                const valStr = M.eval(Variables[row][col]).toString();
                Solution.set(varName, parseInt(valStr, 10));
            })
        );
        return Solution;
    } else if (result === 'unsat') {
        console.log('The problem is not solvable.');
    } else {
        console.log('Z3 cannot determine whether the problem is solvable.');
    }
}

In [ ]:
console.time("Solver Duration");
const Solution = await solve();
console.timeEnd("Solver Duration");
Solution

## Graphical Representation

The function `show_solution` displays the given solution on a 9x9 grid, rendering it directly to SVG using `tslab`.

In [ ]:
import * as tslab from "tslab";
import { RecursiveMap as Map } from "recursive-set";

function show_solution(Solution: Map<string, number> | undefined, width: string = "50%"): void {
    if (!Solution) {
        console.log("No solution to display.");
        return;
    }
    const Sudoku = create_puzzle();
    const cellSize = 50;
    const totalSize = 9 * cellSize;
    
    let html = `<svg width="${width}" viewBox="-5 -5 ${totalSize + 10} ${totalSize + 10}" xmlns="http://www.w3.org/2000/svg" style="font-family: sans-serif;">`;
    
    html += range(9).flatMap(r => 
        range(9).map(c => {
            const x = c * cellSize;
            const y = r * cellSize;
            const isGiven = Sudoku[r][c] !== '*';
            const val = isGiven ? Sudoku[r][c] : Solution.get(`V${r + 1}${c + 1}`);
            const bg = isGiven ? '#f0f0f0' : '#ffffff';
            const color = isGiven ? '#000000' : '#1a73e8';
            const weight = isGiven ? 'bold' : 'normal';
            
            let cellHtml = `<rect x="${x}" y="${y}" width="${cellSize}" height="${cellSize}" fill="${bg}" stroke="#cccccc" stroke-width="1" />`;
            if (val !== undefined) {
                cellHtml += `<text x="${x + cellSize / 2}" y="${y + cellSize / 2 + 7}" text-anchor="middle" font-size="22" fill="${color}" font-weight="${weight}">${val}</text>`;
            }
            return cellHtml;
        })
    ).join('');
    
    html += [0, 3, 6, 9].map(i => {
        const pos = i * cellSize;
        return `<line x1="0" y1="${pos}" x2="${totalSize}" y2="${pos}" stroke="#333333" stroke-width="3" stroke-linecap="square" />` +
               `<line x1="${pos}" y1="0" x2="${pos}" y2="${totalSize}" stroke="#333333" stroke-width="3" stroke-linecap="square" />`;
    }).join('');
    
    html += `</svg>`;
    tslab.display.html(html);
}

In [ ]:
show_solution(Solution);